# 🏛️ Notebook 3: A Central Config Server (with hot reload)

So far each service reads its *own* env vars or config file. In a real
microservices system you often have **dozens** of services that need to
share settings (timeouts, feature flags, URLs, limits).

The **Config Server pattern**:

```
┌────────────┐    GET /config/orders     ┌────────────────┐
│ orders svc │ ────────────────────────► │ config server  │
│ pay svc    │ ────────────────────────► │  (central DB)  │
│ ship svc   │ ────────────────────────► │                │
└────────────┘                           └────────────────┘
```

Real examples: **Spring Cloud Config**, **HashiCorp Consul**,
**etcd**, **AWS AppConfig**, **Kubernetes ConfigMaps**.


## 🛠️ Setup

```bash
cd 05-microservices/configuration-externalization
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## 🟩 A mini config server

We'll simulate a central server with a plain Python class and have two
"services" pull from it. No network — just to show the shape of the pattern.


In [ ]:
import time, threading

class ConfigServer:
    """Central store. Versioned so clients can tell when config changes."""
    def __init__(self):
        self._data: dict[str, dict] = {}
        self._version = 0
        self._lock = threading.Lock()

    def put(self, service: str, cfg: dict):
        with self._lock:
            self._data[service] = dict(cfg)
            self._version += 1
            print(f"[server] {service} updated -> v{self._version}")

    def get(self, service: str) -> tuple[int, dict]:
        with self._lock:
            return self._version, dict(self._data.get(service, {}))

server = ConfigServer()
server.put("orders", {"timeout_ms": 500, "retries": 3})
server.put("payments", {"timeout_ms": 2000, "provider": "stripe"})
print(server.get("orders"))
print(server.get("payments"))


## 🔄 Hot reload from the client side

A service **polls** the server (or subscribes) and updates its in-memory
config when the version changes. No restart needed.


In [ ]:
class ServiceClient:
    def __init__(self, name: str, server: ConfigServer):
        self.name = name
        self.server = server
        self.version = -1
        self.config: dict = {}
        self.refresh()

    def refresh(self):
        v, cfg = self.server.get(self.name)
        if v != self.version:
            self.version = v
            self.config = cfg
            print(f"[{self.name}] reloaded config v{v}: {cfg}")

    def handle_request(self, req: str) -> str:
        self.refresh()  # a real app might poll every N seconds instead
        return f"{self.name} handled {req!r} with timeout={self.config.get('timeout_ms')}ms"

orders = ServiceClient("orders", server)
print(orders.handle_request("buy"))

# Ops bumps the timeout live — no redeploy:
server.put("orders", {"timeout_ms": 1500, "retries": 5})
print(orders.handle_request("buy"))


## 📣 Push-based: watchers / subscribers

Polling is simple but either laggy (long interval) or wasteful (short
interval). Real tools like **etcd**, **Consul**, and **ZooKeeper** let
clients **subscribe** — the server pushes only when something changes.

Here's the same pattern with plain callbacks.


In [ ]:
class PushConfigServer(ConfigServer):
    def __init__(self):
        super().__init__()
        self._subscribers: dict[str, list] = {}

    def subscribe(self, service: str, callback):
        self._subscribers.setdefault(service, []).append(callback)

    def put(self, service: str, cfg: dict):
        super().put(service, cfg)
        for cb in self._subscribers.get(service, []):
            cb(self._version, dict(cfg))

push_server = PushConfigServer()

class PushClient:
    def __init__(self, name, server):
        self.name, self.server = name, server
        self.config: dict = {}
        server.subscribe(name, self._on_change)

    def _on_change(self, version, cfg):
        self.config = cfg
        print(f"[{self.name}] pushed v{version}: {cfg}")

orders = PushClient("orders", push_server)
push_server.put("orders", {"timeout_ms": 500})
push_server.put("orders", {"timeout_ms": 2000})   # arrives instantly, no polling


### 🔁 Pull vs push — which to use?

| | Pull (poll) | Push (subscribe) |
|--|--|--|
| Client complexity | very low | needs a persistent connection |
| Latency to change | up to the poll interval | ~instant |
| Server load | constant | proportional to changes |
| Works behind strict firewalls | yes (outbound HTTP) | sometimes harder |

Most real systems combine them: **long-poll / SSE / gRPC streams** so the
client still initiates the connection (firewall-friendly) but the server
only replies when something changes.


## 🛟 What happens if the config server is down?

Rule: **a config outage must not take your services down.**
Clients should cache the last-known-good config and keep serving traffic.


In [ ]:
class ResilientClient(ServiceClient):
    def refresh(self):
        try:
            v, cfg = self.server.get(self.name)
            if v != self.version:
                self.version = v
                self.config = cfg
                print(f"[{self.name}] reloaded v{v}")
        except Exception as e:                  # e.g. network error
            print(f"[{self.name}] config server down ({e}); using cached v{self.version}")

# Break the server and make sure the client keeps working.
class BrokenServer:
    def get(self, service): raise ConnectionError("config server unreachable")

rc = ResilientClient("orders", server)   # fetch good config first
rc.server = BrokenServer()                # now simulate outage
print(rc.handle_request("buy"))           # still uses cached config


## 💣 The failure mode a config server *creates*

Everything above makes config changes fast and global. That is the feature — and it is
also the risk, because **a bad config now reaches every service in seconds, with none
of the safety rails a code deploy has**: no canary, no CI, no gradual rollout, no code
review. Several of the largest public internet outages of the last decade were a config
push, not a code push.

Here is the whole disaster in twelve lines.

In [ ]:
bad_server = PushConfigServer()

class Worker(PushClient):
    def __init__(self, name, server, service):
        self.service = service
        super().__init__(name, server)
        self.healthy = True

    def _on_change(self, version, cfg):
        self.config = cfg
        # A real service *applies* config — and applying it can fail.
        timeout = cfg.get("timeout_ms")
        self.healthy = isinstance(timeout, int) and timeout > 0

fleet = [Worker(f"orders-{i}", bad_server, "orders") for i in range(8)]
bad_server.subscribe("orders", lambda v, c: None)   # (server keyed by service name)

# We have to fan the push out to the whole fleet — that's what a config server does.
for w in fleet:
    bad_server.subscribe("orders", w._on_change)

bad_server.put("orders", {"timeout_ms": 500})
print(f"good config -> healthy workers: {sum(w.healthy for w in fleet)}/8")

# 3am: someone fixes a "typo" in the config UI. Note it is valid JSON, valid YAML,
# and completely wrong.
bad_server.put("orders", {"timeout_ms": "500ms"})
print(f"bad config  -> healthy workers: {sum(w.healthy for w in fleet)}/8  💥")
print()
print("Eight instances, one keystroke, zero deploys, and no rollback in flight.")
print("The blast radius of a config server is the entire fleet, instantly.")

### Three rails that make this survivable

1. **Validate at the server, before storing.** The config service should reject a change
   that fails the schema, so the bad value never reaches a single client. This is exactly
   what AWS AppConfig's validators and Kubernetes admission webhooks are for.
2. **Roll it out in stages.** Push to 1 instance, watch health, then 10%, then the rest.
   A config change deserves the same canary a code change gets.
3. **Let clients reject and keep the last-known-good.** A client that cannot *apply* new
   config must keep serving with the old one and shout — never adopt config it can't use.

All three, together:

In [ ]:
from pydantic import BaseModel, Field, ValidationError

class OrdersConfig(BaseModel):
    timeout_ms: int = Field(ge=1, le=60_000)
    retries: int = Field(default=3, ge=0, le=10)

class ValidatingConfigServer(PushConfigServer):
    """Rail 1 + 2: validate before storing, then push in stages."""
    def put_staged(self, service, cfg, subscribers, stages=(1, 3, None)):
        try:
            OrdersConfig(**cfg)                       # rail 1
        except ValidationError as e:
            print(f"  🛑 server REJECTED the config: {e.errors()[0]['msg']}")
            return False
        for n in stages:                              # rail 2
            targets = subscribers if n is None else subscribers[:n]
            for w in targets:
                w._on_change(0, dict(cfg))
            unhealthy = [w.name for w in targets if not w.healthy]
            label = 'all' if n is None else n
            if unhealthy:
                print(f"  ⏹  stage {label}: {len(unhealthy)} unhealthy — halting rollout")
                return False
            print(f"  ✅ stage {label}: {len(targets)} instance(s) healthy")
        return True

class SafeWorker(Worker):
    """Rail 3: never adopt config you cannot apply."""
    def _on_change(self, version, cfg):
        try:
            OrdersConfig(**cfg)
        except ValidationError:
            print(f"    [{self.name}] refusing invalid config, keeping last-known-good")
            return
        self.config, self.healthy = cfg, True

srv = ValidatingConfigServer()
fleet = [SafeWorker(f"orders-{i}", srv, "orders") for i in range(8)]
for w in fleet:
    w.config, w.healthy = {"timeout_ms": 500}, True

print("pushing the SAME bad config through the safe path:")
srv.put_staged("orders", {"timeout_ms": "500ms"}, fleet)
print(f"  healthy workers: {sum(w.healthy for w in fleet)}/8  ✅ nobody was affected")

print("\npushing a good config:")
srv.put_staged("orders", {"timeout_ms": 1500, "retries": 5}, fleet)
print(f"  healthy workers: {sum(w.healthy for w in fleet)}/8")

## 🧭 When a config server earns its keep — and when it doesn't

### ✅ Use one when
- You have **many services** that share settings, and updating them means editing many
  places today.
- You need to change behaviour **without a restart** (timeouts, limits, kill switches).
- You need an **audit trail** of who changed what, when — config changes cause outages
  and you will want the history.

### 🚫 Skip it when
- You have a handful of services. Env vars + a restart is simpler, safer, and already
  works. A config server is a **new tier-0 dependency**; you now have to run it, secure
  it, and page someone when it's down.
- The value only changes at **deploy time**. That is not config-server material — that
  is an env var, and a redeploy is a perfectly good delivery mechanism with better
  rollback than any config UI.
- You'd be storing **secrets** in it without encryption and per-key access control.
  Use a secrets manager (Notebook 4).
- Your clients **cannot** tolerate the server being unreachable. If you can't cache
  last-known-good, you have converted "config service has a bad minute" into
  "everything is down".

## 🏗️ Real-world shapes of this pattern

| Tool | Style | Notes |
|---|---|---|
| **Spring Cloud Config** | HTTP + git backend | Classic JVM microservices setup |
| **Consul / etcd** | KV store + watch | Used as a primitive by many systems |
| **Kubernetes ConfigMap/Secret** | Mounted as files or env | Restart pod *or* use a sidecar reloader |
| **AWS AppConfig / Parameter Store** | Managed service | Validation + staged rollout built-in |
| **LaunchDarkly / Unleash** | Flags-focused | Great for per-user targeting (see NB 2) |

## 🧠 Takeaways

- Centralising config scales *much* better than per-service env vars.
- Always **version** config so clients know when to reload.
- Clients must tolerate the server being **temporarily unreachable**.
- Combine with feature flags (NB 2) and secrets (NB 4) for a complete picture.
